# SpaceX Falcon 9 - Data Collection (API)

Collect historical Falcon 9 launch records directly from the public SpaceX REST API (`api.spacexdata.com`). We flatten the nested JSON (rocket, payload, launchpad and landing-core references) into one launch-level table.

In [ ]:
import requests
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

## Request launch data from the SpaceX v4 API

In [ ]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print(response.status_code)
data = pd.json_normalize(response.json())
data.head()

## Flatten nested reference IDs

Each launch references a `rocket`, `payloads`, `launchpad` and `cores` id. We look each of those up via their own endpoints and build flat columns: `BoosterVersion`, `PayloadMass`, `Orbit`, `LaunchSite`, `Outcome`, `Flights`, `GridFins`, `Reused`, `Legs`, `LandingPad`, `Block`, `ReusedCount`, `Serial`, `Longitude`, `Latitude`.

In [ ]:
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= pd.to_datetime('2020-11-13').date()]
data.reset_index(drop=True, inplace=True)
print(f"{len(data)} single-core, single-payload Falcon launches retained")
data.head()

In [ ]:
BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Legs = [], [], [], [], []
LandingPad, Block, ReusedCount, Serial, Longitude, Latitude = [], [], [], [], [], []

def getBoosterVersion(data):
    for x in data['rocket']:
        response = requests.get(f"https://api.spacexdata.com/v4/rockets/{x}").json()
        BoosterVersion.append(response['name'])

def getLaunchSite(data):
    for x in data['launchpad']:
        response = requests.get(f"https://api.spacexdata.com/v4/launchpads/{x}").json()
        Longitude.append(response['longitude'])
        Latitude.append(response['latitude'])
        LaunchSite.append(response['name'])

def getPayloadData(data):
    for load in data['payloads']:
        response = requests.get(f"https://api.spacexdata.com/v4/payloads/{load}").json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get(f"https://api.spacexdata.com/v4/cores/{core['core']}").json()
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None); ReusedCount.append(None); Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

In [ ]:
getBoosterVersion(data)
getLaunchSite(data)
getPayloadData(data)
getCoreData(data)
print("Lookup complete:", len(BoosterVersion), "booster versions,", len(LaunchSite), "launch sites")

In [ ]:
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}
df = pd.DataFrame(launch_dict)
df.head()

## Keep Falcon 9 only and export

In [ ]:
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1']
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(data_falcon9['PayloadMass'].mean())
print(data_falcon9.isnull().sum())
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print("Saved dataset_part_1.csv:", data_falcon9.shape)